In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import csv
import os
import re
import time

CSV_FILE = "aljazeera.csv"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/115.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.aljazeera.com/",
}

def load_existing_urls():
    """Load already scraped URLs from CSV for deduplication."""
    if not os.path.exists(CSV_FILE):
        return set()
    with open(CSV_FILE, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        return {row["url"] for row in reader}

def get_article_links(base_url):
    """Get all unique Al Jazeera article links from a section page."""
    try:
        resp = requests.get(base_url, headers=HEADERS, timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        links = set()
        for a in soup.find_all("a", href=True):
            href = a["href"]

            # Ensure absolute URL
            full_url = urljoin("https://www.aljazeera.com", href)

            # Match URLs with /YYYY/ pattern
            if re.search(r"/\d{4}/", full_url) and "/news/" in full_url:
                links.add(full_url)

        return sorted(links)

    except Exception as e:
        print(f"Error fetching {base_url}: {e}")
        return []


def scrape_article_aljazeera(url):
    """Scrape the main Al Jazeera article body."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        # Main content container
        content_div = soup.find("div", class_="wysiwyg wysiwyg--all-content")
        if not content_div:
            return "", "", ""

        parts = []
        for tag in content_div.find_all(["p", "li"]):
            text = tag.get_text(strip=True)
            if text:
                parts.append(text)

        # Title
        title_tag = soup.find("h1")
        title = title_tag.get_text(strip=True) if title_tag else ""

        # Date
        published_date = soup.find("div", class_="date-simple").find_all("span")[1].get_text(strip=True)
        if not published_date:
            published_date = soup.find("time")["datetime"] if soup.find("time") else ""
        return title, published_date, "\n".join(parts)

    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return "", "", ""


def save_articles(articles):
    """Save scraped articles to CSV (append mode)."""
    file_exists = os.path.exists(CSV_FILE)
    with open(CSV_FILE, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["title", "url", "published_date", "content"]
        )
        if not file_exists:
            writer.writeheader()
        for article in articles:
            writer.writerow(article)


if __name__ == "__main__":
    base_urls = [
        "https://www.aljazeera.com/opinions/",
        "https://www.aljazeera.com/us-canada/",
        "https://www.aljazeera.com/tag/explainer/"
    ]

    existing_urls = load_existing_urls()
    all_links = set()

    for url in base_urls:
        section_links = get_article_links(url)
        print(f"Found {len(section_links)} from {url}")
        all_links.update(section_links)
        time.sleep(5)  # be polite

    print(f"Total unique candidate articles: {len(all_links)}")

    scraped_data = []
    i =0
    for link in all_links:
        if link in existing_urls:
            print(f"Skipping (already in CSV): {link}")
            continue
        time.sleep(10)  # be polite
        print(f"Scraping: {link}")
        title, published_date, content = scrape_article_aljazeera(link)
        if content:
            scraped_data.append({
                "title": title,
                "url": link,
                "published_date": published_date,
                "content": content
            })
        i += 1
        if i >= 3:  # Limit to first 5 articles for testing
            break
        
    if scraped_data:
        save_articles(scraped_data)
        print(f"Saved {len(scraped_data)} new articles.")
    else:
        print("No new articles found.")


Found 19 from https://www.aljazeera.com/opinions/
Found 19 from https://www.aljazeera.com/us-canada/
Found 19 from https://www.aljazeera.com/tag/explainer/
Total unique candidate articles: 54
Scraping: https://www.aljazeera.com/opinions/2025/8/5/the-airdrops-on-gaza-are-a-pr-stunt-not-a-humanitarian-operation
Scraping: https://www.aljazeera.com/news/2025/8/8/us-appeals-court-throws-out-trump-contempt-ruling-over-deportation-flights
Scraping: https://www.aljazeera.com/opinions/2025/7/31/former-eu-ambassadors-warn-europes-silence-on-gaza-is-complicity
Saved 3 new articles.
